# Musanga V4 — NB04 V4.004: safe MS metadata and limited native-window QA

The original MS data have full-width, **single-row GeoTIFF strips** (except tiled Baego), so raster-wide previews and random scatter samples can cause excessive memory use. **Do not use WarpedVRT or raster-wide downsampling.**

Cells 00–03 read metadata only. **New cell 04** inspects at most three small *64×64 native-resolution windows* per MS raster, one file at a time, with GDAL caching capped. It measures both bands separately (including possible transparency/alpha), without assuming radiometric calibration.

No U-Net training, no raw datasets modified, no full mosaic in RAM. All new outputs write to `Outputs/v4/04_ms_qa_v4_004`. Native-window results are samples only, not full ROI quality metrics.


In [ ]:
# 00 — Reuse NB00 inventory and frozen ROI; metadata only, no image read
from pathlib import Path
import json, math, re, warnings
import numpy as np, pandas as pd, geopandas as gpd, rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio.features import rasterize
from rasterio.warp import transform_bounds
from rasterio.vrt import WarpedVRT
from rasterio.transform import from_bounds
from scipy.ndimage import distance_transform_edt
import matplotlib.pyplot as plt
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').is_dir(): drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final')
DATA=ROOT/'Data'
NB00=ROOT/'Outputs/v4/00_input_audit'
OUT=ROOT/'Outputs/v4/04_ms_qa_v4_004'
OUT.mkdir(parents=True,exist_ok=True)
REG=NB00/'00_ms_registry.json'
HEADERS=NB00/'00_ms_raster_headers.csv'
OVERLAP=NB00/'00_ms_rgb_extent_audit.csv'
assert REG.is_file() and HEADERS.is_file() and OVERLAP.is_file(),(
    'First run NB00 MS inventory cells; missing',REG,HEADERS,OVERLAP)
ms_manifest=json.loads(REG.read_text())
ms=pd.read_csv(HEADERS).fillna({'error':''})
overlap=pd.read_csv(OVERLAP)
ROI_PATH=DATA/'Ortho_RGB'/'ROI.shp'
roi=gpd.read_file(ROI_PATH)
assert roi.crs is not None and len(roi)>0
RGB_REFERENCE=ms_manifest['rgb_reference']
RGB_DIR=DATA/'Ortho_RGB'
RGB_FILES={group:RGB_DIR/name for group,name in RGB_REFERENCE.items()}
assert all(p.is_file() for p in RGB_FILES.values()),RGB_FILES
EDGE_DISTANCE_M=(1,3,5,10,20,30,50)
print('Available surveys:', sorted(ms.group.dropna().unique()))
print('NB00 MS records:',len(ms),'| paired extent records:',len(overlap))
display(ms[['group','band_hint','name','bands','dtype','nodata','error']].head(40))


In [ ]:
# 01 — Band headers only, ZERO source pixel reads
rows=[]
for rec in ms.itertuples():
    if getattr(rec,'error',''):continue
    path=Path(rec.path)
    if not path.is_file():continue
    with rasterio.Env(GDAL_CACHEMAX=32):
        with rasterio.open(path) as ds:
            for b in range(1,ds.count+1):
                rows.append(dict(group=rec.group,file=path.name,path=str(path),
                    band=b,band_hint=rec.band_hint,
                    description=ds.descriptions[b-1],
                    colorinterp=str(ds.colorinterp[b-1]),
                    dtype=ds.dtypes[b-1],scale=ds.scales[b-1],offset=ds.offsets[b-1],
                    nodata=ds.nodatavals[b-1],crs=str(ds.crs),
                    width=ds.width,height=ds.height,
                    pixel_x_m=abs(ds.transform.a),pixel_y_m=abs(ds.transform.e),
                    block_shape=str(ds.block_shapes[b-1]),
                    pixel_distribution='NOT SAMPLED',file_bytes=path.stat().st_size))
    print('HEADERS:',rec.group,rec.name,flush=True)
BAND_AUDIT=pd.DataFrame(rows)
assert not BAND_AUDIT.empty
BAND_AUDIT.to_csv(OUT/'04_ms_band_metadata_only.csv',index=False)
display(BAND_AUDIT[['group','file','band','band_hint','description','dtype',
                    'scale','offset','nodata','pixel_x_m','block_shape']])
print('No pixel data read, no band-value statistics inferred.')


In [ ]:
# 02 — ROI / RGB / MS BOUNDING-BOX overlap ONLY, ZERO pixel reads
from shapely.geometry import box
from shapely.ops import unary_union
from rasterio.warp import transform_bounds
summary=[]
for group,records in ms.groupby('group',sort=True):
    if group not in RGB_FILES:continue
    with rasterio.open(RGB_FILES[group]) as rgb:
        study=unary_union(roi.to_crs(rgb.crs).geometry).intersection(box(*rgb.bounds))
        area=float(study.area)
        if area<=0:continue
        for rec in records.itertuples():
            if getattr(rec,'error',''):continue
            path=Path(rec.path)
            if not path.is_file():continue
            with rasterio.Env(GDAL_CACHEMAX=32):
                with rasterio.open(path) as src:
                    if src.crs is None:continue
                    bb=box(*transform_bounds(src.crs,rgb.crs,*src.bounds,densify_pts=21))
                    frac=float(study.intersection(bb).area/area)
                    summary.append(dict(group=group,file=rec.name,band_hint=rec.band_hint,
                        nominal_bands=int(src.count),roi_bbox_overlap_fraction=frac,
                        rgb_roi_area_ha=area/1e4,valid_pixel_coverage='NOT TESTED',
                        registration='NOT TESTED'))
            print('BBOX:',group,rec.name,round(frac,3),flush=True)
SUMMARY=pd.DataFrame(summary)
assert not SUMMARY.empty
SUMMARY.to_csv(OUT/'04_ms_rgb_roi_bbox_only.csv',index=False)
display(SUMMARY.sort_values(['group','file']).round(3))
print('Bounding-box overlap is only an upper limit on valid MS pixel coverage.')


In [ ]:
# 03 — Metadata coverage figure, NO pixel reads
d=SUMMARY.sort_values(['group','file'])
fig,ax=plt.subplots(figsize=(9,max(4,len(d)*.23)))
ax.barh(np.arange(len(d)),d.roi_bbox_overlap_fraction.to_numpy())
ax.set_yticks(np.arange(len(d)))
ax.set_yticklabels([f"{r.group}: {r.file[:40]}" for r in d.itertuples()],fontsize=7)
ax.invert_yaxis()
ax.set_xlim(0,1.02)
ax.set_xlabel('Bounding-box overlap with RGB ∩ ROI (not actual valid coverage)')
ax.set_title('MS inventory — metadata only')
fig.tight_layout()
fig.savefig(OUT/'04_ms_bbox_metadata_summary.png',dpi=120,bbox_inches='tight')
plt.show()
plt.close(fig)
print('DONE:',OUT)


In [ ]:
# 04 — STRICTLY BOUNDED native-window MS sampling; do not resample whole images.
# 25 files x at most 3 windows x <= 2/4 bands x 64x64 pixels.
# Full-width strip TIFFs may decode an entire scanline internally; never request
# hundreds of rows, use WarpedVRT, or keep open handles/arrays between files.
from rasterio.warp import transform as transform_coords
from rasterio.windows import Window
from shapely.geometry import Point,box
from shapely.ops import unary_union
import gc

WINDOW_SIZE=64
MAX_PATCHES_PER_FILE=3
QA_ROWS=[]
QA_PATCHES=[]
QA_WARNINGS=[]
for group,records in ms.groupby('group',sort=True):
    if group not in RGB_FILES:continue
    with rasterio.open(RGB_FILES[group]) as rgb:
        crs=rgb.crs
        study=unary_union(roi.to_crs(crs).geometry).intersection(box(*rgb.bounds))
        if study.is_empty:continue
        for rec in records.itertuples():
            if getattr(rec,'error',''):continue
            path=Path(rec.path)
            if not path.is_file():continue
            print('NATIVE WINDOWS:',group,path.name,flush=True)
            # Only one source open; cap GDAL's decompressed block cache at 16 MB.
            with rasterio.Env(GDAL_CACHEMAX=16):
                with rasterio.open(path) as src:
                    if src.crs is None:continue
                    from rasterio.warp import transform_bounds
                    source_box=box(*transform_bounds(src.crs,crs,*src.bounds,densify_pts=21))
                    relevant=study.intersection(source_box)
                    if relevant.is_empty:continue
                    bb=relevant.bounds
                    choices=[relevant.representative_point()]
                    # Deterministic geographic candidates, no raster samples/read.
                    for u,v in [(0.25,0.25),(0.75,0.75),(0.25,0.75),
                                (0.75,0.25),(0.5,0.5),(0.35,0.65),(0.65,0.35)]:
                        pt=Point(bb[0]+u*(bb[2]-bb[0]),bb[1]+v*(bb[3]-bb[1]))
                        if relevant.covers(pt):choices.append(pt)
                    seen=set()
                    count=0
                    for pt in choices:
                        if count>=MAX_PATCHES_PER_FILE:break
                        if src.crs==crs:x,y=pt.x,pt.y
                        else:
                            mx,my=transform_coords(crs,src.crs,[pt.x],[pt.y])
                            x,y=mx[0],my[0]
                        row,col=src.index(x,y)
                        if row<0 or col<0 or row>=src.height or col>=src.width:continue
                        col=int(np.clip(col-WINDOW_SIZE//2,0,max(0,src.width-WINDOW_SIZE)))
                        row=int(np.clip(row-WINDOW_SIZE//2,0,max(0,src.height-WINDOW_SIZE)))
                        key=(row//WINDOW_SIZE,col//WINDOW_SIZE)
                        if key in seen:continue
                        seen.add(key)
                        win=Window(col,row,min(WINDOW_SIZE,src.width-col),
                                   min(WINDOW_SIZE,src.height-row))
                        # Never read all bands at once; no large intermediate arrays.
                        pixel_arrays=[]
                        for band in range(1,src.count+1):
                            a=src.read(band,window=win,masked=True)
                            v=np.asarray(a.data)
                            valid=(~np.ma.getmaskarray(a))&np.isfinite(v)
                            vals=v[valid]
                            unique=np.unique(v) if v.size<=4096 else np.array([])
                            QA_ROWS.append(dict(group=group,file=path.name,
                                band_hint=rec.band_hint,band=band,patch=count,
                                row=row,col=col,n_window=int(v.size),
                                n_valid=int(valid.sum()),
                                valid_fraction=float(valid.mean()),
                                min=float(vals.min()) if len(vals) else np.nan,
                                p05=float(np.percentile(vals,5)) if len(vals) else np.nan,
                                median=float(np.median(vals)) if len(vals) else np.nan,
                                p95=float(np.percentile(vals,95)) if len(vals) else np.nan,
                                max=float(vals.max()) if len(vals) else np.nan,
                                zero_frac_valid=float(np.mean(vals==0)) if len(vals) else np.nan,
                                fraction_65535=float(np.mean(vals==65535)) if len(vals) else np.nan,
                                n_unique=int(len(unique)),
                                unique_values=str(unique[:12].tolist()) if len(unique)<=12 else 'many'))
                            # Caution: 2-band TIFF may be intensity + alpha. Report, do
                            # not pre-assign alpha without a diagnostic.
                            del a,v,valid,vals
                        count+=1
                    if count==0:QA_WARNINGS.append((group,path.name,'No in-footprint native patch'))
            print('  COMPLETED:',count,'native 64x64 patches; GDAL cache capped',flush=True)
            gc.collect()

PATCH_QA=pd.DataFrame(QA_ROWS)
assert not PATCH_QA.empty,'No valid sample windows in any MS raster'
PATCH_QA.to_csv(OUT/'04_native_patch_band_statistics.csv',index=False)
summ=(PATCH_QA.groupby(['group','file','band_hint','band'],dropna=False)
      .agg(n_patches=('patch','nunique'),mean_valid=('valid_fraction','mean'),
           median_p05=('p05','median'),median=('median','median'),
           median_p95=('p95','median'),mean_zero=('zero_frac_valid','mean'),
           mean_65535=('fraction_65535','mean'),
           median_n_unique=('n_unique','median')).reset_index())
summ.to_csv(OUT/'04_native_patch_summary.csv',index=False)
display(summ.round(3))
if QA_WARNINGS:print('NO VALID WINDOWS:',QA_WARNINGS)
print('Sample distributions are NOT proof of calibrated reflectance or registration.')
print('Native-window audit complete:',OUT)


## Next decision gate

Review the 64×64 native-window statistics, particularly intensity-versus-alpha behavior in the two-band UINT16 rasters, duplicate MOS 1–2 red-edge candidates, and the suspected MOS 5–6 bands. Brightness distributions are **uncalibrated DN until verified**. These spot samples do not measure actual ROI-wide coverage or solve RGB-MS registration.

Only then implement **ROI-local multi-band matched patch comparisons**, with per-acquisition masks, neighborhood-relative indices and spatial holdouts. Avoid generic full-resolution reproject/resample operations on strip-organized files.
